<a href="https://colab.research.google.com/github/mic006016/seoul-bigdata-2026/blob/main/seoul_bigdata.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder

# ==========================================
# 1. SKT 유동인구 데이터 전처리
# ==========================================
skt = pd.read_csv('SKT 월별 요일별 유동인구.csv')

# 1-1) 컬럼명 수정
skt = skt.rename(columns={
    '기준년월(STD_YM)': 'std_ym',
    'X좌표(X_COORD)': 'x_coord',
    'Y좌표(Y_COORD)': 'y_coord',
    '성별코드(GNDR_CD)': 'gndr_cd',
    '연령대코드(AGE_GR_SCTN_CD)': 'age_cd',
    '요일코드(WKDY_CD)': 'wkdy_cd',
    '유동인구수(FLOW_POP_CNT)': 'flow_pop_cnt'
})

# 1-2) 불필요한 컬럼 삭제 (자치구, 시간대코드)
skt = skt.drop(columns=['자치구(SIGUNGU)', '시간대코드(TMST_CD)'], errors='ignore')

# 1-3) 연령대코드 5년 -> 10년 간격으로 통합 함수
def map_skt_age(age):
    if pd.isna(age): return np.nan
    age = int(age)
    if age < 2000: return '1019'       # 0509, 1014, 1519 -> 1019 (20세 미만)
    elif age < 3000: return '2029'     # 2024, 2529 -> 2029
    elif age < 4000: return '3039'
    elif age < 5000: return '4049'
    elif age < 6000: return '5059'
    elif age < 7000: return '6069'
    else: return '7099'                # 70세 이상 통합

skt['age_cd'] = skt['age_cd'].apply(map_skt_age)

# 1-4) 시간대를 삭제했으므로 중복 행을 그룹화하여 유동인구 합산 (메모리 다이어트)
skt_grouped = skt.groupby(
    ['std_ym', 'x_coord', 'y_coord', 'gndr_cd', 'age_cd', 'wkdy_cd']
)['flow_pop_cnt'].sum().reset_index()


# ==========================================
# 2. 신한카드 매출 데이터 전처리
# ==========================================
card = pd.read_csv('3.서울시민의 성별 연령대별(격자별).csv')

# 2-1) 컬럼명 수정
card = card.rename(columns={
    '기준일자': 'std_date',
    '격자_250': 'grid_250',
    '개인법인구분': 'pvt_corp',
    '성별': 'gndr_cd',
    '연령대': 'age_cd',
    '업종대분류': 'industry',
    '카드이용금액계': 'sales_amount',
    '카드이용건수계': 'sales_count'
})

# 결측치 제거 (성별, 연령대가 없는 법인카드 등은 분석 목적상 제외하거나 별도 처리 필요)
card = card.dropna(subset=['gndr_cd', 'age_cd'])

# 2-2) 기준년월(std_ym) 컬럼 생성 및 요일코드(wkdy_cd) 추출
# 날짜형식 변환 후 추출 (SKT의 요일코드와 동일하게 1=월요일 ~ 7=일요일로 맞춤)
card['std_date_dt'] = pd.to_datetime(card['std_date'], format='%Y%m%d')
card['std_ym'] = card['std_date'] // 100
card['wkdy_cd'] = card['std_date_dt'].dt.dayofweek + 1

# 2-3) 라벨인코딩 및 매핑
# 성별은 SKT와 맞추기 위해 수동 매핑 (남=1, 여=2)
card['gndr_cd'] = card['gndr_cd'].map({'남': 1, '여': 2})

# 개인법인구분, 업종대분류는 LabelEncoder 사용
le_pvt = LabelEncoder()
le_ind = LabelEncoder()
card['pvt_corp'] = le_pvt.fit_transform(card['pvt_corp'])
card['industry'] = le_ind.fit_transform(card['industry'])

# 나중에 디코딩을 위해 매핑 딕셔너리 출력 저장 추천
print("업종 라벨 매핑:", dict(zip(le_ind.classes_, le_ind.transform(le_ind.classes_))))

# 2-4) 연령대코드(age_cd) 10년 간격 변환
def map_card_age(age):
    age = str(age)
    if age == '20세미만': return '1019'
    elif age == '70세이상': return '7099'
    else: return age[:2] + age[3:5] # '60_69세' -> '60' + '69' -> '6069'

card['age_cd'] = card['age_cd'].apply(map_card_age)

# 2-5) 불필요한 원본 날짜 컬럼 삭제
card_grouped = card.drop(columns=['std_date', 'std_date_dt'])

# 일별 데이터를 묶었으므로 여기서도 동일 기준으로 합산 (메모리 다이어트)
card_grouped = card_grouped.groupby(
    ['std_ym', 'grid_250', 'pvt_corp', 'gndr_cd', 'age_cd', 'industry', 'wkdy_cd']
)[['sales_amount', 'sales_count']].sum().reset_index()


# 결과 확인
print("SKT 전처리 완료 데이터 셋:", skt_grouped.shape)
print("카드 전처리 완료 데이터 셋:", card_grouped.shape)